## NYISO leakage-safe load forecasts

This section selects the Hudson Valley load-forecast vintage that was available
when each NYISO day-ahead prediction would have been made.

For an electricity-delivery day \(D\), the prediction cutoff is defined as
5:00 a.m. Eastern Time on \(D-1\). Forecast vintages timestamped after this
cutoff are excluded to prevent the model from using information that would not
have been available at prediction time.

In [ ]:
from pathlib import Path

import pandas as pd

# Starting from the current working folder, search upward through parent folders
# until the project root is found. The project root is identified by pyproject.toml.

def find_project_root(start: Path| None= None) -> Path:
    """Find the repository directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    # Stop with a clear error if the notebook is not running within this project.
    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml."
    )

# Resolve the project root so all later file paths are portable.
PROJECT_ROOT = find_project_root()

NYISO_ELECTRICITY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "nyiso_hudson_valley_january_2025_electricity.csv"
)

# Define the cleaned NYISO electricity dataset produced by Notebook 02.
NYISO_FORECAST_VINTAGES_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "nyiso_hudson_valley_load_forecast_vintages.csv"
)

# Confirm both required inputs exist before feature engineering begins.
assert NYISO_ELECTRICITY_FILE.exists()
assert NYISO_FORECAST_VINTAGES_FILE.exists()

# Display the resolved file paths for reproducibility and troubleshooting.
print("Electricity:", NYISO_ELECTRICITY_FILE)
print("Forecasts:", NYISO_FORECAST_VINTAGES_FILE)

Electricity: c:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\nyiso_hudson_valley_january_2025_electricity.csv
Forecasts: c:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\interim\nyiso_hudson_valley_load_forecast_vintages.csv


### Load the electricity and forecast-vintage data

The processed NYISO electricity data contain the hourly day-ahead LBMP target.
The interim forecast table contains all six Hudson Valley load-forecast vintages
for every January 2025 target hour.

Timestamp columns are parsed explicitly so availability comparisons are made
using timezone-aware values.

In [ ]:
# Load the cleaned NYISO electricity data and all archived load-forecast
# vintages. Parse timestamps explicitly as timezone-aware values so forecast
# availability can be compared safely with each prediction cutoff.

nyiso_electricity = pd.read_csv(
    NYISO_ELECTRICITY_FILE
)

nyiso_electricity = nyiso_electricity.rename(
    columns={
        "day_ahead_price_usd_mwh": "day_ahead_lmp",
    }
)

forecast_vintages = pd.read_csv(
    NYISO_FORECAST_VINTAGES_FILE
)

nyiso_electricity["timestamp_utc"] = pd.to_datetime(
    nyiso_electricity["timestamp_utc"],
    utc=True,
)

nyiso_electricity["timestamp_local"] = (
    pd.to_datetime(
        nyiso_electricity["timestamp_local"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

forecast_vintages["target_timestamp"] = (
    pd.to_datetime(
        forecast_vintages["target_timestamp"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

forecast_vintages["forecast_available_at"] = (
    pd.to_datetime(
        forecast_vintages["forecast_available_at"],
        utc=True,
    )
    .dt.tz_convert("America/New_York")
)

print("Electricity rows:", len(nyiso_electricity))
print("Forecast-vintage rows:", len(forecast_vintages))
print(
    "Unique forecast target hours:",
    forecast_vintages["target_timestamp"].nunique(),
)

Electricity rows: 744
Forecast-vintage rows: 4464
Unique forecast target hours: 744


### Calculate the day-ahead prediction cutoff

For every target operating day, the prediction cutoff is 5:00 a.m. Eastern Time
on the preceding calendar day. All 24 target hours belonging to the same
operating day therefore share the same prediction cutoff.

The cutoff is constructed from local calendar dates so the logic remains valid
when the study is later expanded across daylight-saving-time transitions.

In [ ]:
# Calculate one day-ahead prediction cutoff per target operating day:
# 5:00 a.m. Eastern Time on the preceding calendar day. All 24 delivery
# hours on the same operating date share this cutoff.

target_operating_dates = pd.to_datetime(
    forecast_vintages["target_timestamp"].dt.date
)

cutoff_dates = (
    target_operating_dates
    - pd.DateOffset(days=1)
    + pd.Timedelta(hours=5)
)

forecast_vintages["prediction_cutoff"] = (
    cutoff_dates.dt.tz_localize(
        "America/New_York",
        ambiguous="raise",
        nonexistent="raise",
    )
)

forecast_vintages[
    [
        "target_timestamp",
        "forecast_available_at",
        "prediction_cutoff",
        "source_file",
    ]
].head()

,target_timestamp,forecast_available_at,prediction_cutoff,source_file
0,2025-01-01 00:00:00-05:00,2024-12-26 07:05:02-05:00,2024-12-31 05:00:00-05:00,20241227isolf.csv
1,2025-01-01 00:00:00-05:00,2024-12-27 07:50:04-05:00,2024-12-31 05:00:00-05:00,20241228isolf.csv
2,2025-01-01 00:00:00-05:00,2024-12-28 07:50:08-05:00,2024-12-31 05:00:00-05:00,20241229isolf.csv
3,2025-01-01 00:00:00-05:00,2024-12-29 07:30:06-05:00,2024-12-31 05:00:00-05:00,20241230isolf.csv
4,2025-01-01 00:00:00-05:00,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241231isolf.csv


In [ ]:
# Inspect the calculated cutoff for January 10 to confirm that every target
# hour on that operating day uses January 9 at 5:00 a.m. Eastern.

january_10_cutoffs = (
    forecast_vintages.loc[
        forecast_vintages["target_timestamp"].dt.date
        == pd.Timestamp("2025-01-10").date(),
        "prediction_cutoff",
    ]
    .drop_duplicates()
)

january_10_cutoffs

1296   2025-01-09 05:00:00-05:00
Name: prediction_cutoff, dtype: datetime64[us, America/New_York]

In [ ]:
# Validate the cutoff calculation: retain all forecast-vintage rows, cover
# all 744 January target hours, and confirm the expected January 10 cutoff.

assert len(forecast_vintages) == 4464
assert forecast_vintages["target_timestamp"].nunique() == 744
assert forecast_vintages["prediction_cutoff"].notna().all()

assert january_10_cutoffs.tolist() == [
    pd.Timestamp(
        "2025-01-09 05:00",
        tz="America/New_York",
    )
]

print("Step 4 passed: prediction cutoffs calculated correctly.")

Step 4 passed: prediction cutoffs calculated correctly.


### Identify forecast vintages available by the cutoff

A forecast vintage is eligible only when its recorded availability timestamp is
at or before the 5:00 a.m. Eastern Time prediction cutoff for the target
operating day.

Forecasts published after the cutoff are excluded because they would not have
been available when the day-ahead prediction was made.

In [ ]:
# Keep only forecast vintages that were available at or before each target
# hour's prediction cutoff. This prevents later forecast revisions from
# leaking future information into the day-ahead model.

eligible_vintages = forecast_vintages.loc[
    forecast_vintages["forecast_available_at"]
    <= forecast_vintages["prediction_cutoff"]
].copy()

eligible_counts = (
    eligible_vintages
    .groupby("target_timestamp")
    .size()
)

assert eligible_counts.size == 744
assert eligible_counts.ge(1).all()

assert (
    eligible_vintages["forecast_available_at"]
    <= eligible_vintages["prediction_cutoff"]
).all()

print("Eligible forecast rows:", len(eligible_vintages))
print(
    "Eligible vintages per target hour:",
    eligible_counts.value_counts().sort_index().to_dict(),
)

Eligible forecast rows: 3720
Eligible vintages per target hour: {5: 744}


### Select the latest eligible forecast

For each target operating hour, the eligible vintage with the most recent
availability timestamp is selected.

A tie at the latest availability timestamp is treated as a data-quality error,
rather than being silently resolved. The selected rows retain their forecast,
source, and availability provenance for auditability.

In [ ]:
# For each target hour, select the most recent forecast that was still
# available by the cutoff. Preserve source and availability fields so the
# selected forecast can be audited later.

eligible_vintages["latest_eligible_available_at"] = (
    eligible_vintages
    .groupby("target_timestamp")["forecast_available_at"]
    .transform("max")
)

latest_eligible_rows = eligible_vintages.loc[
    eligible_vintages["forecast_available_at"]
    == eligible_vintages["latest_eligible_available_at"]
].copy()

latest_row_counts = (
    latest_eligible_rows
    .groupby("target_timestamp")
    .size()
)

assert latest_row_counts.size == 744
assert latest_row_counts.eq(1).all(), (
    "Expected exactly one latest eligible vintage per target hour; "
    f"found ties for {latest_row_counts[latest_row_counts.ne(1)].index.tolist()}"
)

selected_forecast_columns = [
    "target_timestamp",
    "load_forecast_mw",
    "forecast_available_at",
    "prediction_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
]

if "forecast_horizon_hours" in latest_eligible_rows.columns:
    selected_forecast_columns.append("forecast_horizon_hours")

selected_load_forecasts = (
    latest_eligible_rows
    .loc[:, selected_forecast_columns]
    .sort_values("target_timestamp")
    .reset_index(drop=True)
)

selected_load_forecasts["hours_before_cutoff"] = (
    selected_load_forecasts["prediction_cutoff"]
    - selected_load_forecasts["forecast_available_at"]
).dt.total_seconds() / 3600

selected_load_forecasts[
    [
        "target_timestamp",
        "load_forecast_mw",
        "forecast_available_at",
        "prediction_cutoff",
        "hours_before_cutoff",
        "source_archive",
        "source_file",
    ]
].head()

,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,hours_before_cutoff,source_archive,source_file
0,2025-01-01 00:00:00-05:00,930,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
1,2025-01-01 01:00:00-05:00,889,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
2,2025-01-01 02:00:00-05:00,855,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
3,2025-01-01 03:00:00-05:00,841,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv
4,2025-01-01 04:00:00-05:00,839,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,21.332222,20241201isolf_csv.zip,20241231isolf.csv


### Verify the selected forecast vintages

The selected dataset should contain one forecast for each of the 744 January
target hours. Every selected forecast must have been available by its prediction
cutoff.

January 10 is inspected separately because its latest source report,
`20250110isolf.csv`, became available after the January 9 cutoff. The expected
leakage-safe source is therefore `20250109isolf.csv`.

In [ ]:
# Validate that exactly one eligible forecast was selected for every January
# target hour and that no selected forecast was available after its cutoff.
# Confirm January 10 correctly uses the January 9 source report.

january_10_selected = selected_load_forecasts.loc[
    selected_load_forecasts["target_timestamp"].dt.date
    == pd.Timestamp("2025-01-10").date(),
    [
        "target_timestamp",
        "load_forecast_mw",
        "forecast_available_at",
        "prediction_cutoff",
        "hours_before_cutoff",
        "source_file",
    ],
]

january_10_selected.head()

,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,hours_before_cutoff,source_file
216,2025-01-10 00:00:00-05:00,1117,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
217,2025-01-10 01:00:00-05:00,1091,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
218,2025-01-10 02:00:00-05:00,1077,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
219,2025-01-10 03:00:00-05:00,1074,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv
220,2025-01-10 04:00:00-05:00,1086,2025-01-08 07:35:04-05:00,2025-01-09 05:00:00-05:00,21.415556,20250109isolf.csv


In [ ]:
# Confirm electricity hours and selected forecast target hours align exactly,
# then merge one leakage-safe load forecast into each NYISO electricity row.
# A one-to-one merge prevents duplicated or unmatched delivery hours.

assert len(selected_load_forecasts) == 744

assert selected_load_forecasts["target_timestamp"].nunique() == 744

assert not selected_load_forecasts.duplicated(
    "target_timestamp"
).any()

assert selected_load_forecasts["load_forecast_mw"].notna().all()

assert (
    selected_load_forecasts["forecast_available_at"]
    <= selected_load_forecasts["prediction_cutoff"]
).all()

assert selected_load_forecasts["hours_before_cutoff"].ge(0).all()

expected_latest_times = (
    eligible_vintages
    .groupby("target_timestamp")["forecast_available_at"]
    .max()
    .sort_index()
)

actual_latest_times = (
    selected_load_forecasts
    .set_index("target_timestamp")["forecast_available_at"]
    .sort_index()
)

pd.testing.assert_series_equal(
    actual_latest_times,
    expected_latest_times,
    check_names=False,
)

assert len(january_10_selected) == 24

assert set(january_10_selected["source_file"]) == {
    "20250109isolf.csv"
}

print("Step 5 passed: one latest eligible forecast selected for each target hour.")

Step 5 passed: one latest eligible forecast selected for each target hour.


## Step 6: Merge the leakage-safe load forecasts

Both `target_timestamp` and `timestamp_local` represent the beginning of the
same America/New_York delivery hour. Their exact one-to-one correspondence is
validated before the merge.

The left merge preserves all 744 NYISO electricity target hours and carries the
selected forecast's cutoff and source-provenance fields forward for auditability.

In [ ]:
# Validate the merged modeling dataframe: 744 unique hourly rows, matching
# delivery timestamps, nonmissing selected forecasts, and complete forecast
# provenance showing every forecast was available by its prediction cutoff.

electricity_target_hours = pd.DatetimeIndex(
    nyiso_electricity["timestamp_local"]
).sort_values()
forecast_target_hours = pd.DatetimeIndex(
    selected_load_forecasts["target_timestamp"]
).sort_values()

assert electricity_target_hours.tz is not None
assert forecast_target_hours.tz is not None

pd.testing.assert_index_equal(
    electricity_target_hours,
    forecast_target_hours,
    check_names=False,
)

nyiso_electricity_with_load_forecasts = (
    nyiso_electricity
    .merge(
        selected_load_forecasts,
        how="left",
        left_on="timestamp_local",
        right_on="target_timestamp",
        validate="one_to_one",
    )
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

nyiso_electricity_with_load_forecasts.head()

,timestamp_local,timestamp_utc,location_id,location,day_ahead_price_usd_mwh,loss_component_usd_mwh,congestion_component_usd_mwh,energy_component_usd_mwh,source_time_zone,load_location,...,target_timestamp,load_forecast_mw,forecast_available_at,prediction_cutoff,source_archive,source_file,availability_basis,availability_is_proxy,forecast_horizon_hours,hours_before_cutoff
0,2025-01-01 00:00:00-05:00,2025-01-01 05:00:00+00:00,61758,HUD VL,33.16,1.31,0.0,31.85,EST,HUD VL,...,2025-01-01 00:00:00-05:00,930,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,40.332222,21.332222
1,2025-01-01 01:00:00-05:00,2025-01-01 06:00:00+00:00,61758,HUD VL,32.07,1.26,0.0,30.81,EST,HUD VL,...,2025-01-01 01:00:00-05:00,889,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,41.332222,21.332222
2,2025-01-01 02:00:00-05:00,2025-01-01 07:00:00+00:00,61758,HUD VL,30.02,1.16,0.0,28.86,EST,HUD VL,...,2025-01-01 02:00:00-05:00,855,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,42.332222,21.332222
3,2025-01-01 03:00:00-05:00,2025-01-01 08:00:00+00:00,61758,HUD VL,28.28,1.01,0.0,27.27,EST,HUD VL,...,2025-01-01 03:00:00-05:00,841,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,43.332222,21.332222
4,2025-01-01 04:00:00-05:00,2025-01-01 09:00:00+00:00,61758,HUD VL,28.22,1.03,0.0,27.19,EST,HUD VL,...,2025-01-01 04:00:00-05:00,839,2024-12-30 07:40:04-05:00,2024-12-31 05:00:00-05:00,20241201isolf_csv.zip,20241231isolf.csv,zip_entry_last_modified,True,44.332222,21.332222


In [11]:
assert len(nyiso_electricity_with_load_forecasts) == 744
assert nyiso_electricity_with_load_forecasts["timestamp_local"].is_unique
assert nyiso_electricity_with_load_forecasts["target_timestamp"].is_unique

pd.testing.assert_series_equal(
    nyiso_electricity_with_load_forecasts["timestamp_local"],
    nyiso_electricity_with_load_forecasts["target_timestamp"],
    check_names=False,
)

assert nyiso_electricity_with_load_forecasts["load_forecast_mw"].notna().all()

forecast_audit_columns = [
    "forecast_available_at",
    "prediction_cutoff",
    "hours_before_cutoff",
    "source_archive",
    "source_file",
    "availability_basis",
    "availability_is_proxy",
]

assert set(forecast_audit_columns).issubset(
    nyiso_electricity_with_load_forecasts.columns
)

assert (
    nyiso_electricity_with_load_forecasts["forecast_available_at"]
    <= nyiso_electricity_with_load_forecasts["prediction_cutoff"]
).all()

print("Step 6 passed: leakage-safe load forecasts merged one-to-one with NYISO electricity.")

Step 6 passed: leakage-safe load forecasts merged one-to-one with NYISO electricity.
